# Brain Stroke CT Detection Using Transfer Learning

**Dataset:** Brain Stroke CT Dataset  
**Classes:** Normal, Ischemia, Bleeding  
**Default backbone:** EfficientNetB0

This notebook contains the complete project in one `.ipynb` file:
- Data handling and EDA
- Image preprocessing and augmentation
- Scaling and feature engineering
- Feature selection
- Histograms, box plots, scatter plots, bar charts, heatmaps, pairplots, categorical and regression plots
- PCA and K-Means clustering
- CNN + transfer learning
- EfficientNetB0, MobileNetV2, ResNet50, VGG16 and DenseNet121 support
- Early stopping and fine-tuning
- Training curves
- Test-set evaluation
- Confusion matrices
- ROC/AUC
- Error visualization
- Single-image prediction
- Saving the trained model and results


In [ ]:
# ============================================================
# 1. INSTALL / IMPORT
# ============================================================

!pip -q install kagglehub seaborn scikit-learn pillow statsmodels

import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    roc_auc_score,
    precision_recall_fscore_support
)

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)
from tensorflow.keras.preprocessing.image import (
    ImageDataGenerator,
    load_img,
    img_to_array
)
from tensorflow.keras.applications import (
    MobileNetV2,
    EfficientNetB0,
    ResNet50,
    VGG16,
    DenseNet121
)
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))


## 2. Configuration

The notebook downloads the Kaggle dataset automatically. The dataset URL/slug is the one supplied for the project.

If you already have the dataset extracted, you can set `DATASET_DIR` manually instead.


In [ ]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = 224
BATCH_SIZE = 16

# Initial transfer-learning epochs
EPOCHS = 30

# Fine-tuning epochs
FINE_TUNE_EPOCHS = 15

NUM_CLASSES = 3

CLASS_NAMES = [
    "Normal",
    "Ischemia",
    "Bleeding"
]

# Main model requested for the project
BACKBONE_NAME = "EfficientNetB0"

# Other supported choices:
# "MobileNetV2"
# "EfficientNetB0"
# "ResNet50"
# "VGG16"
# "DenseNet121"

OUTPUT_DIR = "/content/brain_stroke_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Leave as None for automatic Kaggle download.
# If you already extracted the dataset, replace with its folder.
DATASET_DIR = None

print("Backbone:", BACKBONE_NAME)
print("Image size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Output:", OUTPUT_DIR)


## 3. Download the Kaggle dataset

The preferred route in Colab is `kagglehub`, so you do not need to manually download and upload the ZIP.

If the automatic download produces a directory with an extra nested folder, the helper below searches for the class folders.


In [ ]:
# ============================================================
# 3. DOWNLOAD DATASET
# ============================================================

if DATASET_DIR is None:
    import kagglehub

    downloaded_path = kagglehub.dataset_download(
        "ozguraslank/brain-stroke-ct-dataset"
    )

    print("Downloaded dataset path:")
    print(downloaded_path)

    DATASET_ROOT = downloaded_path
else:
    DATASET_ROOT = DATASET_DIR

print("Dataset root:", DATASET_ROOT)


In [ ]:
# ============================================================
# 4. FIND CLASS FOLDERS
# ============================================================

def find_folder(root, target):
    matches = []

    for current_root, dirs, files in os.walk(root):
        for d in dirs:
            if d.lower() == target.lower():
                matches.append(os.path.join(current_root, d))

    return matches[0] if matches else None

class_directories = {}

for class_name in CLASS_NAMES:
    folder = find_folder(DATASET_ROOT, class_name)
    class_directories[class_name] = folder
    print(f"{class_name}: {folder}")

missing = [c for c, p in class_directories.items() if p is None]

if missing:
    raise FileNotFoundError(
        "Could not find class folders: " + ", ".join(missing)
    )


## 5. Data collection and class distribution


In [ ]:
# ============================================================
# 5. COLLECT IMAGE FILES
# ============================================================

IMAGE_EXTENSIONS = (
    ".jpg", ".jpeg", ".png",
    ".bmp", ".tif", ".tiff"
)

records = []

for label, class_name in enumerate(CLASS_NAMES):

    folder = class_directories[class_name]

    for root, dirs, files in os.walk(folder):

        for filename in files:

            if filename.lower().endswith(IMAGE_EXTENSIONS):

                records.append({
                    "filepath": os.path.join(root, filename),
                    "label": label,
                    "class": class_name
                })

df = pd.DataFrame(records)

print("Total images:", len(df))
display(df.head())

print("\nClass distribution:")
display(df["class"].value_counts().rename_axis("class").reset_index(name="count"))


In [ ]:
# ============================================================
# 6. BAR CHART - CLASS DISTRIBUTION
# ============================================================

plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="class", order=CLASS_NAMES)
plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/class_distribution.png", dpi=300)
plt.show()


## 6. Sample images


In [ ]:
# ============================================================
# 7. SAMPLE IMAGE VISUALIZATION
# ============================================================

fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for row, class_name in enumerate(CLASS_NAMES):

    class_df = df[df["class"] == class_name]

    samples = class_df.sample(
        min(3, len(class_df)),
        random_state=SEED
    )

    for col, (_, sample) in enumerate(samples.iterrows()):

        image = Image.open(sample["filepath"]).convert("RGB")

        axes[row, col].imshow(image)
        axes[row, col].set_title(class_name)
        axes[row, col].axis("off")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/sample_images.png", dpi=300)
plt.show()


## 7. Feature engineering for EDA, scaling, feature selection and clustering

These handcrafted image statistics are **not the main CNN features**. They provide numerical features for the traditional data-science requirements in the assignment.


In [ ]:
# ============================================================
# 8. IMAGE FEATURE ENGINEERING
# ============================================================

def extract_image_features(path):

    image = Image.open(path).convert("L")
    image = image.resize((128, 128))

    arr = np.asarray(image, dtype=np.float32) / 255.0
    pixels = arr.flatten()

    return {
        "mean_intensity": np.mean(pixels),
        "std_intensity": np.std(pixels),
        "min_intensity": np.min(pixels),
        "max_intensity": np.max(pixels),
        "median_intensity": np.median(pixels),
        "q25_intensity": np.percentile(pixels, 25),
        "q75_intensity": np.percentile(pixels, 75),
        "dark_pixel_ratio": np.mean(pixels < 0.20),
        "bright_pixel_ratio": np.mean(pixels > 0.80),
        "edge_strength": np.mean(np.abs(np.diff(arr, axis=0))),
        "horizontal_variance": np.var(np.mean(arr, axis=0)),
        "vertical_variance": np.var(np.mean(arr, axis=1))
    }

feature_records = []

for _, row in df.iterrows():

    features = extract_image_features(row["filepath"])
    features["class"] = row["class"]
    features["label"] = row["label"]

    feature_records.append(features)

features_df = pd.DataFrame(feature_records)

print("Engineered feature shape:", features_df.shape)
display(features_df.head())


## 8. Histograms


In [ ]:
# ============================================================
# 9. HISTOGRAMS
# ============================================================

numeric_features = [
    "mean_intensity",
    "std_intensity",
    "median_intensity",
    "dark_pixel_ratio",
    "bright_pixel_ratio",
    "edge_strength"
]

for feature in numeric_features:

    plt.figure(figsize=(8, 5))

    sns.histplot(
        data=features_df,
        x=feature,
        hue="class",
        kde=True,
        bins=30
    )

    plt.title(f"Histogram of {feature}")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/histogram_{feature}.png", dpi=300)
    plt.show()


## 9. Box plots


In [ ]:
# ============================================================
# 10. BOX PLOTS
# ============================================================

for feature in numeric_features:

    plt.figure(figsize=(8, 5))

    sns.boxplot(
        data=features_df,
        x="class",
        y=feature,
        order=CLASS_NAMES
    )

    plt.title(f"Box Plot - {feature}")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/boxplot_{feature}.png", dpi=300)
    plt.show()


## 10. Scatter, categorical and regression plots


In [ ]:
# ============================================================
# 11. SCATTER PLOT
# ============================================================

plt.figure(figsize=(8, 6))

sns.scatterplot(
    data=features_df,
    x="mean_intensity",
    y="std_intensity",
    hue="class",
    style="class",
    alpha=0.7
)

plt.title("Mean Intensity vs Standard Deviation")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/scatter_plot.png", dpi=300)
plt.show()


# ============================================================
# 12. CATEGORICAL PLOT
# ============================================================

plt.figure(figsize=(9, 6))

sns.violinplot(
    data=features_df,
    x="class",
    y="mean_intensity",
    order=CLASS_NAMES
)

plt.title("Mean Intensity by Class")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/categorical_plot.png", dpi=300)
plt.show()


# ============================================================
# 13. REGRESSION PLOT
# ============================================================

plt.figure(figsize=(8, 6))

sns.regplot(
    data=features_df,
    x="mean_intensity",
    y="std_intensity",
    scatter_kws={"alpha": 0.35}
)

plt.title("Regression Plot: Mean vs Standard Deviation")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/regression_plot.png", dpi=300)
plt.show()


## 11. Heatmap and pairplot


In [ ]:
# ============================================================
# 14. CORRELATION HEATMAP
# ============================================================

plt.figure(figsize=(12, 9))

correlation_matrix = features_df[numeric_features].corr()

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/feature_correlation_heatmap.png", dpi=300)
plt.show()


# ============================================================
# 15. PAIRPLOT
# ============================================================

pairplot_features = [
    "mean_intensity",
    "std_intensity",
    "median_intensity",
    "edge_strength"
]

g = sns.pairplot(
    features_df[pairplot_features + ["class"]],
    hue="class",
    diag_kind="hist"
)

g.fig.suptitle(
    "Pairplot of Engineered Features",
    y=1.02
)

g.savefig(f"{OUTPUT_DIR}/pairplot.png", dpi=300)
plt.show()


## 12. Scaling and feature selection


In [ ]:
# ============================================================
# 16. SCALING
# ============================================================

X_features = features_df[numeric_features]
y_features = features_df["label"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_features)

print("Scaled feature matrix:", X_scaled.shape)


# ============================================================
# 17. FEATURE SELECTION
# ============================================================

k = min(5, X_scaled.shape[1])

selector = SelectKBest(
    score_func=f_classif,
    k=k
)

X_selected = selector.fit_transform(
    X_scaled,
    y_features
)

selected_feature_names = [
    feature
    for feature, selected
    in zip(numeric_features, selector.get_support())
    if selected
]

feature_scores = pd.DataFrame({
    "feature": numeric_features,
    "score": selector.scores_,
    "selected": selector.get_support()
}).sort_values("score", ascending=False)

print("Selected features:")
for feature in selected_feature_names:
    print("-", feature)

display(feature_scores)

plt.figure(figsize=(10, 6))

sns.barplot(
    data=feature_scores,
    x="score",
    y="feature"
)

plt.title("ANOVA Feature Selection Scores")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/feature_selection.png", dpi=300)
plt.show()


## 13. PCA and K-Means clustering


In [ ]:
# ============================================================
# 18. PCA
# ============================================================

pca = PCA(n_components=2, random_state=SEED)

X_pca = pca.fit_transform(X_selected)

pca_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "class": features_df["class"].values
})

plt.figure(figsize=(9, 7))

sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="class",
    style="class",
    s=70,
    alpha=0.75
)

plt.title("PCA Visualization")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/pca_visualization.png", dpi=300)
plt.show()


# ============================================================
# 19. K-MEANS CLUSTERING
# ============================================================

kmeans = KMeans(
    n_clusters=NUM_CLASSES,
    random_state=SEED,
    n_init=10
)

clusters = kmeans.fit_predict(X_selected)

cluster_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "cluster": clusters,
    "class": features_df["class"].values
})

plt.figure(figsize=(9, 7))

sns.scatterplot(
    data=cluster_df,
    x="PC1",
    y="PC2",
    hue="cluster",
    palette="viridis",
    s=70
)

plt.title("K-Means Clustering")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/kmeans_clusters.png", dpi=300)
plt.show()


## 14. Train/validation/test split

We use a stratified 70/15/15 split so that the three classes are represented proportionally.


In [ ]:
# ============================================================
# 20. DATA SPLIT
# ============================================================

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    stratify=df["label"]
)

validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Training:", len(train_df))
print("Validation:", len(validation_df))
print("Testing:", len(test_df))

print("\nTraining distribution:")
display(train_df["class"].value_counts())

print("\nValidation distribution:")
display(validation_df["class"].value_counts())

print("\nTesting distribution:")
display(test_df["class"].value_counts())


## 15. Data augmentation and generators

Augmentation is applied **only to the training set**. Validation and test images are not randomly augmented.


In [ ]:
# ============================================================
# 21. DATA AUGMENTATION
# ============================================================

train_datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.10,
    shear_range=0.05,
    horizontal_flip=True,
    fill_mode="nearest"
)

validation_datagen = ImageDataGenerator()
test_datagen = ImageDataGenerator()


# ============================================================
# 22. GENERATORS
# ============================================================

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col="filepath",
    y_col="class",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="categorical",
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

validation_generator = validation_datagen.flow_from_dataframe(
    dataframe=validation_df,
    x_col="filepath",
    y_col="class",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="categorical",
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col="filepath",
    y_col="class",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="categorical",
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Generator class indices:")
print(train_generator.class_indices)


## 16. Transfer-learning backbone

The default is **EfficientNetB0**, as requested. You can change one variable in the configuration cell to compare other pretrained CNNs.


In [ ]:
# ============================================================
# 23. PRETRAINED MODEL SELECTION
# ============================================================

def get_backbone(name):

    name = name.lower()

    input_shape = (IMG_SIZE, IMG_SIZE, 3)

    if name == "mobilenetv2":
        backbone = MobileNetV2(
            include_top=False,
            weights="imagenet",
            input_shape=input_shape
        )
        preprocess = mobilenet_preprocess

    elif name == "efficientnetb0":
        backbone = EfficientNetB0(
            include_top=False,
            weights="imagenet",
            input_shape=input_shape
        )
        preprocess = efficientnet_preprocess

    elif name == "resnet50":
        backbone = ResNet50(
            include_top=False,
            weights="imagenet",
            input_shape=input_shape
        )
        preprocess = resnet_preprocess

    elif name == "vgg16":
        backbone = VGG16(
            include_top=False,
            weights="imagenet",
            input_shape=input_shape
        )
        preprocess = vgg_preprocess

    elif name == "densenet121":
        backbone = DenseNet121(
            include_top=False,
            weights="imagenet",
            input_shape=input_shape
        )
        preprocess = densenet_preprocess

    else:
        raise ValueError(
            "Choose one of: MobileNetV2, EfficientNetB0, "
            "ResNet50, VGG16, DenseNet121"
        )

    return backbone, preprocess


backbone, preprocess_function = get_backbone(
    BACKBONE_NAME
)

print("Loaded:", BACKBONE_NAME)
print("Number of backbone layers:", len(backbone.layers))


## 17. CNN architecture

Architecture:

**Input → Preprocessing → Pretrained CNN → Global Average Pooling → Dense(ReLU) → Batch Normalization → Dropout → Dense(Softmax)**

The pretrained convolutional base is initially frozen.


In [ ]:
# ============================================================
# 24. CNN + TRANSFER LEARNING MODEL
# ============================================================

backbone.trainable = False

inputs = layers.Input(
    shape=(IMG_SIZE, IMG_SIZE, 3),
    name="input_image"
)

x = layers.Lambda(
    preprocess_function,
    name="preprocessing"
)(inputs)

x = backbone(
    x,
    training=False
)

x = layers.GlobalAveragePooling2D(
    name="global_average_pooling"
)(x)

x = layers.Dense(
    256,
    activation="relu",
    name="dense_feature_layer"
)(x)

x = layers.BatchNormalization(
    name="batch_normalization"
)(x)

x = layers.Dropout(
    0.40
)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax",
    name="classification_output"
)(x)

model = Model(
    inputs=inputs,
    outputs=outputs
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


## 18. Early stopping, checkpointing and training


In [ ]:
# ============================================================
# 25. CALLBACKS
# ============================================================

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=6,
    restore_best_weights=True,
    verbose=1
)

checkpoint = ModelCheckpoint(
    filepath=os.path.join(
        OUTPUT_DIR,
        "best_brain_stroke_model.keras"
    ),
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=3,
    min_lr=1e-7,
    verbose=1
)


# ============================================================
# 26. INITIAL TRANSFER LEARNING
# ============================================================

history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=EPOCHS,
    callbacks=[
        early_stopping,
        checkpoint,
        reduce_lr
    ]
)


## 19. Fine-tuning

After training the new classification head, the last ~30% of the pretrained backbone is unfrozen and trained using a much smaller learning rate.


In [ ]:
# ============================================================
# 27. FINE-TUNING
# ============================================================

backbone.trainable = True

fine_tune_from = int(
    len(backbone.layers) * 0.70
)

for layer in backbone.layers[:fine_tune_from]:
    layer.trainable = False

# Keep BatchNormalization layers frozen for more stable fine-tuning.
for layer in backbone.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

fine_tune_history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=[
        early_stopping,
        checkpoint,
        reduce_lr
    ]
)


## 20. Training curves


In [ ]:
# ============================================================
# 28. COMBINE TRAINING HISTORIES
# ============================================================

acc = history.history["accuracy"] + fine_tune_history.history["accuracy"]
val_acc = history.history["val_accuracy"] + fine_tune_history.history["val_accuracy"]

loss = history.history["loss"] + fine_tune_history.history["loss"]
val_loss = history.history["val_loss"] + fine_tune_history.history["val_loss"]


# ============================================================
# 29. ACCURACY CURVE
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(acc, label="Training Accuracy")
plt.plot(val_acc, label="Validation Accuracy")

plt.title("Training and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/accuracy_curve.png", dpi=300)
plt.show()


# ============================================================
# 30. LOSS CURVE
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(loss, label="Training Loss")
plt.plot(val_loss, label="Validation Loss")

plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/loss_curve.png", dpi=300)
plt.show()


## 21. Evaluate on the test set


In [ ]:
# ============================================================
# 31. LOAD BEST MODEL
# ============================================================

best_model_path = os.path.join(
    OUTPUT_DIR,
    "best_brain_stroke_model.keras"
)

if os.path.exists(best_model_path):
    model = tf.keras.models.load_model(best_model_path)
    print("Best model loaded.")


# ============================================================
# 32. TEST SET EVALUATION
# ============================================================

test_loss, test_accuracy = model.evaluate(
    test_generator,
    verbose=1
)

print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)


# ============================================================
# 33. PREDICTIONS
# ============================================================

test_generator.reset()

probabilities = model.predict(
    test_generator,
    verbose=1
)

predicted_labels = np.argmax(
    probabilities,
    axis=1
)

true_labels = test_generator.classes


## 22. Classification report and per-class metrics


In [ ]:
# ============================================================
# 34. CLASSIFICATION REPORT
# ============================================================

report = classification_report(
    true_labels,
    predicted_labels,
    target_names=CLASS_NAMES,
    digits=4
)

print(report)

with open(
    f"{OUTPUT_DIR}/classification_report.txt",
    "w"
) as f:
    f.write(report)


# ============================================================
# 35. PRECISION / RECALL / F1
# ============================================================

precision, recall, f1, support = precision_recall_fscore_support(
    true_labels,
    predicted_labels,
    average=None,
    labels=range(NUM_CLASSES)
)

metrics_df = pd.DataFrame({
    "Class": CLASS_NAMES,
    "Precision": precision,
    "Recall": recall,
    "F1 Score": f1,
    "Support": support
})

display(metrics_df)


## 23. Confusion matrix


In [ ]:
# ============================================================
# 36. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    true_labels,
    predicted_labels
)

plt.figure(figsize=(8, 7))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/confusion_matrix.png", dpi=300)
plt.show()


# ============================================================
# 37. NORMALIZED CONFUSION MATRIX
# ============================================================

cm_normalized = cm.astype(float) / cm.sum(
    axis=1,
    keepdims=True
)

plt.figure(figsize=(8, 7))

sns.heatmap(
    cm_normalized,
    annot=True,
    fmt=".2f",
    cmap="Greens",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)

plt.title("Normalized Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")

plt.tight_layout()
plt.savefig(
    f"{OUTPUT_DIR}/normalized_confusion_matrix.png",
    dpi=300
)
plt.show()


## 24. Multi-class ROC curve and AUC


In [ ]:
# ============================================================
# 38. MULTI-CLASS ROC
# ============================================================

true_labels_binary = label_binarize(
    true_labels,
    classes=np.arange(NUM_CLASSES)
)

plt.figure(figsize=(9, 7))

for i, class_name in enumerate(CLASS_NAMES):

    fpr, tpr, _ = roc_curve(
        true_labels_binary[:, i],
        probabilities[:, i]
    )

    roc_auc = auc(fpr, tpr)

    plt.plot(
        fpr,
        tpr,
        label=f"{class_name} AUC = {roc_auc:.4f}"
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)

plt.title("Multi-Class ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/roc_curve.png", dpi=300)
plt.show()


# ============================================================
# 39. MACRO ROC-AUC
# ============================================================

macro_auc = roc_auc_score(
    true_labels_binary,
    probabilities,
    multi_class="ovr",
    average="macro"
)

print("Macro ROC-AUC:", macro_auc)


## 25. Error analysis and visualization


In [ ]:
# ============================================================
# 40. ERROR ANALYSIS
# ============================================================

errors = []

for i in range(len(test_df)):

    actual = true_labels[i]
    predicted = predicted_labels[i]

    if actual != predicted:

        errors.append({
            "filepath": test_df.iloc[i]["filepath"],
            "actual": CLASS_NAMES[actual],
            "predicted": CLASS_NAMES[predicted],
            "confidence": probabilities[i][predicted]
        })

errors_df = pd.DataFrame(errors)

print("Number of incorrect predictions:", len(errors_df))

display(errors_df.head(20))


# ============================================================
# 41. VISUALIZE MISCLASSIFIED IMAGES
# ============================================================

if len(errors_df) > 0:

    number_to_display = min(12, len(errors_df))
    samples = errors_df.head(number_to_display)

    columns = 4
    rows = int(np.ceil(number_to_display / columns))

    fig, axes = plt.subplots(
        rows,
        columns,
        figsize=(16, rows * 4)
    )

    axes = np.array(axes).reshape(rows, columns)

    for i, (_, row) in enumerate(samples.iterrows()):

        image = Image.open(
            row["filepath"]
        ).convert("RGB")

        r = i // columns
        c = i % columns

        axes[r, c].imshow(image)

        axes[r, c].set_title(
            f"Actual: {row['actual']}\n"
            f"Predicted: {row['predicted']}\n"
            f"Confidence: {row['confidence']:.2f}"
        )

        axes[r, c].axis("off")

    for i in range(
        number_to_display,
        rows * columns
    ):
        r = i // columns
        c = i % columns
        axes[r, c].axis("off")

    plt.tight_layout()
    plt.savefig(
        f"{OUTPUT_DIR}/prediction_errors.png",
        dpi=300
    )
    plt.show()
else:
    print("No misclassified test images.")


In [ ]:
# ============================================================
# 42. ERROR TYPE BAR CHART
# ============================================================

if len(errors_df) > 0:

    error_types = (
        errors_df
        .groupby(["actual", "predicted"])
        .size()
        .reset_index(name="count")
    )

    error_types["error_type"] = (
        error_types["actual"]
        + " -> "
        + error_types["predicted"]
    )

    plt.figure(figsize=(10, 6))

    sns.barplot(
        data=error_types,
        x="error_type",
        y="count"
    )

    plt.title("Prediction Error Types")
    plt.xlabel("Actual -> Predicted")
    plt.ylabel("Number of Errors")
    plt.xticks(rotation=30)

    plt.tight_layout()
    plt.savefig(
        f"{OUTPUT_DIR}/error_types.png",
        dpi=300
    )
    plt.show()


## 26. Prediction confidence


In [ ]:
# ============================================================
# 43. CONFIDENCE DISTRIBUTION
# ============================================================

confidence_scores = np.max(
    probabilities,
    axis=1
)

confidence_df = pd.DataFrame({
    "confidence": confidence_scores,
    "correct": true_labels == predicted_labels
})

plt.figure(figsize=(9, 6))

sns.histplot(
    data=confidence_df,
    x="confidence",
    hue="correct",
    bins=20,
    kde=True
)

plt.title("Prediction Confidence Distribution")
plt.tight_layout()
plt.savefig(
    f"{OUTPUT_DIR}/confidence_distribution.png",
    dpi=300
)
plt.show()


## 27. Save all results


In [ ]:
# ============================================================
# 44. SAVE TEST PREDICTIONS
# ============================================================

prediction_df = test_df.copy()

prediction_df["true_class"] = [
    CLASS_NAMES[x] for x in true_labels
]

prediction_df["predicted_class"] = [
    CLASS_NAMES[x] for x in predicted_labels
]

prediction_df["confidence"] = confidence_scores

for i, class_name in enumerate(CLASS_NAMES):

    prediction_df[
        f"probability_{class_name}"
    ] = probabilities[:, i]

prediction_df.to_csv(
    f"{OUTPUT_DIR}/test_predictions.csv",
    index=False
)


# ============================================================
# 45. SAVE ENGINEERED FEATURES
# ============================================================

features_df.to_csv(
    f"{OUTPUT_DIR}/engineered_features.csv",
    index=False
)

feature_scores.to_csv(
    f"{OUTPUT_DIR}/feature_selection_scores.csv",
    index=False
)


# ============================================================
# 46. SAVE FINAL MODEL
# ============================================================

model.save(
    f"{OUTPUT_DIR}/final_brain_stroke_model.keras"
)

print("Results saved to:", OUTPUT_DIR)


## 28. Single-image prediction

Use this after training if you want to test an individual CT image.


In [ ]:
# ============================================================
# 47. SINGLE IMAGE PREDICTION
# ============================================================

def predict_single_image(
    image_path,
    trained_model=model
):

    image = load_img(
        image_path,
        target_size=(IMG_SIZE, IMG_SIZE)
    )

    image_array = img_to_array(image)
    image_array = np.expand_dims(
        image_array,
        axis=0
    )

    probabilities = trained_model.predict(
        image_array,
        verbose=0
    )[0]

    predicted_index = np.argmax(
        probabilities
    )

    predicted_class = CLASS_NAMES[
        predicted_index
    ]

    confidence = probabilities[
        predicted_index
    ]

    print("Prediction:", predicted_class)
    print(
        f"Confidence: {confidence * 100:.2f}%"
    )

    print("\nClass probabilities:")

    for i, class_name in enumerate(CLASS_NAMES):
        print(
            f"{class_name}: "
            f"{probabilities[i] * 100:.2f}%"
        )

    plt.figure(figsize=(6, 6))
    plt.imshow(Image.open(image_path).convert("RGB"))
    plt.title(
        f"Prediction: {predicted_class}\n"
        f"Confidence: {confidence * 100:.2f}%"
    )
    plt.axis("off")
    plt.show()

    return predicted_class, confidence, probabilities


# Example:
#
# TEST_IMAGE = "/content/my_brain_ct.jpg"
# predict_single_image(TEST_IMAGE)


## 29. Final project summary

The notebook has now implemented the requested components:

| Component | Implemented |
|---|---|
| Data handling | ✓ |
| Image preprocessing | ✓ |
| Data augmentation | ✓ |
| Scaling | ✓ |
| Feature engineering | ✓ |
| Feature selection | ✓ |
| Histograms | ✓ |
| Box plots | ✓ |
| Scatter plots | ✓ |
| Bar charts | ✓ |
| Heatmaps | ✓ |
| Pairplot | ✓ |
| Categorical plot | ✓ |
| Regression plot | ✓ |
| PCA | ✓ |
| K-Means clustering | ✓ |
| CNN | ✓ |
| Transfer learning | ✓ |
| EfficientNetB0 | ✓ Default |
| MobileNetV2 | ✓ |
| ResNet50 | ✓ |
| VGG16 | ✓ |
| DenseNet121 | ✓ |
| ReLU activation | ✓ |
| Softmax activation | ✓ |
| Early stopping | ✓ |
| Fine-tuning | ✓ |
| Training curves | ✓ |
| Test evaluation | ✓ |
| Classification report | ✓ |
| Confusion matrix | ✓ |
| ROC curve | ✓ |
| AUC | ✓ |
| Error visualization | ✓ |
| Individual prediction | ✓ |
| Model saving | ✓ |

**Important academic note:** this is an image-classification project for the supplied dataset, not a clinically validated diagnostic system.
